# GNN Model Attribution and Score-Saturation Analysis

**The training and inference pipelines are frozen.** This notebook does not retrain the model,
does not refit any preprocessing, does not change features, and does not change the validated 0.90
threshold. It investigates two questions using the already-frozen model and Class-3 scores:

1. Why are the highest-risk Class-3 transactions receiving extremely high scores, including many
   displayed as 1.0?
2. Which features and/or graph-neighbourhood information contribute to the model output for
   representative high-risk Class-3 transactions?

Language used throughout: "model-based illicit-risk score", "high-risk candidate", "supporting
behavioural indicator", "observed association", "candidate transaction", "feature attribution",
"model sensitivity". Never: "confirmed fraud", "proven illicit", "the feature proves illicit
activity", "the model knows this is fraud", "100% probability of fraud".

## 0. Imports and paths

In [1]:
import time, os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

t0 = time.time()
def log(msg): print(f"[{time.time()-t0:7.1f}s] {msg}", flush=True)

DATA_DIR = r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\raw\eliptic++"
PROCESSED_DIR = r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\processed"
ARTIFACT_DIR = r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\gnn_B_quantile_seed42"
OUTPUTS_DIR = r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs"
PLOTS_DIR = os.path.join(OUTPUTS_DIR, "forensic_analysis_plots")
os.makedirs(PLOTS_DIR, exist_ok=True)
print("Paths configured.")

Paths configured.


## 1. Load frozen model + preprocessing

The GraphSAGE architecture is **read from the saved checkpoint**, not assumed. `forward()` is
reproduced exactly from the frozen training notebook: `conv1 -> ReLU -> dropout -> conv2`.

In [2]:
print("="*70); print("SECTION 1: LOAD FROZEN MODEL + PREPROCESSING"); print("="*70)

checkpoint = torch.load(os.path.join(ARTIFACT_DIR, "model_checkpoint.pt"), weights_only=False)
arch = checkpoint["architecture"]
print("Confirmed architecture from checkpoint (not assumed):", arch)
print("forward() reference (from GraphSAGENet source): conv1 -> ReLU -> dropout -> conv2")

class GraphSAGENet(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, out_dim)
        self.dropout = dropout
    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index); h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.conv2(h, edge_index)

model = GraphSAGENet(in_dim=arch["in_dim"], hidden_dim=arch["hidden_dim"], out_dim=arch["out_dim"], dropout=arch["dropout"])
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
VALIDATED_THRESHOLD = checkpoint["selected_threshold"]
print(f"Model loaded, eval() mode. Validated threshold = {VALIDATED_THRESHOLD}")

qt = joblib.load(os.path.join(ARTIFACT_DIR, "quantile_transformer.joblib"))
median_artifact = joblib.load(os.path.join(ARTIFACT_DIR, "median_imputation.joblib"))
augmented_medians = pd.Series(median_artifact["augmented_medians"])
new_feature_medians = pd.Series(median_artifact["new_feature_medians"])
augmented_features = median_artifact["augmented_features"]
continuous_new = median_artifact["continuous_new_features"]
with open(os.path.join(ARTIFACT_DIR, "feature_columns.json")) as f:
    saved_feature_cols = json.load(f)
with open(os.path.join(ARTIFACT_DIR, "scaling_columns.json")) as f:
    saved_scaling_cols = json.load(f)
with open(os.path.join(ARTIFACT_DIR, "binary_columns.json")) as f:
    saved_binary_cols = json.load(f)
print("Preprocessing artifacts loaded (joblib.load only -- no .fit()).")

SECTION 1: LOAD FROZEN MODEL + PREPROCESSING
Confirmed architecture from checkpoint (not assumed): {'type': 'GraphSAGENet', 'conv1': 'SAGEConv', 'conv2': 'SAGEConv', 'in_dim': 186, 'hidden_dim': 64, 'out_dim': 2, 'dropout': 0.3}
forward() reference (from GraphSAGENet source): conv1 -> ReLU -> dropout -> conv2
Model loaded, eval() mode. Validated threshold = 0.9


Preprocessing artifacts loaded (joblib.load only -- no .fit()).


## 2. Rebuild graph + feature matrix

Identical construction to `05_Inference_Forensic_Analysis.ipynb` / `06_Class3_Forensic_Analysis.ipynb`:
same raw files, same node ordering, same edge list, saved QuantileTransformer applied with
`.transform()` only.

In [3]:
print("\n" + "="*70); print("SECTION 2: REBUILD GRAPH + FEATURE MATRIX"); print("="*70)

tx_features = pd.read_csv(DATA_DIR + r"\txs_features.csv")
tx_classes = pd.read_csv(DATA_DIR + r"\txs_classes.csv")
edges_raw = pd.read_csv(DATA_DIR + r"\txs_edgelist.csv")
tx_data = tx_features.merge(tx_classes, on="txId", how="left")
tx_data = tx_data.sort_values("txId", kind="mergesort").reset_index(drop=True)
assert len(tx_data) == 203769

model_feature_cols = [c for c in tx_features.columns if c != "txId"]
txid_to_node = {txid: i for i, txid in enumerate(tx_data["txId"].values)}
edge_cols = edges_raw.columns.tolist()
src = edges_raw[edge_cols[0]].map(txid_to_node).values
dst = edges_raw[edge_cols[1]].map(txid_to_node).values
edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
assert edge_index.shape[1] == 234355

time_step_arr = tx_data["Time step"].values
def split_of_step(t):
    if t <= 34: return 0
    elif t <= 41: return 1
    else: return 2
split_id = np.array([split_of_step(t) for t in time_step_arr])
known_mask_all = tx_data["class"].isin([1, 2]).values
train_mask_np = (split_id == 0) & known_mask_all
class3_mask_np = (tx_data["class"] == 3).values
assert class3_mask_np.sum() == 157205
print(f"Graph: {len(tx_data)} nodes, {edge_index.shape[1]} edges. Train mask: {train_mask_np.sum()}. Class-3: {class3_mask_np.sum()}.")

REMOVE_13 = [
    "Local_feature_49", "Local_feature_65", "Local_feature_30", "Local_feature_29",
    "Local_feature_23", "Local_feature_47", "Local_feature_25", "Local_feature_43",
    "Local_feature_41", "Local_feature_54", "Local_feature_53", "Local_feature_55",
    "Local_feature_18",
]
a2_cols = [c for c in model_feature_cols if c not in REMOVE_13]
NEW_14 = ["FIN3","FIN4","FIN5","FIN6","STR1","STR2","TMP1","TMP2","TMP4","TMP5","TMP6","TMP7","NET1","NET2"]
new_feats = pd.read_parquet(os.path.join(PROCESSED_DIR, "transaction_time_safe_features_v1.parquet"))
merged = tx_data[["txId"] + a2_cols].merge(new_feats[["txId"] + NEW_14 + ["TMP6_is_first_appearance"]], on="txId", how="left")
merged["augmented_features_missing"] = merged[augmented_features].isna().all(axis=1).astype(int)
final_feature_cols_b = a2_cols + ["augmented_features_missing"] + NEW_14 + ["TMP6_is_first_appearance"]
assert final_feature_cols_b == saved_feature_cols, "STOP: feature ordering mismatch!"
merged_raw = merged.copy()

scaling_cols_b = a2_cols + continuous_new
assert scaling_cols_b == saved_scaling_cols
binary_cols_b = ["augmented_features_missing", "STR2", "TMP6_is_first_appearance"]
assert binary_cols_b == saved_binary_cols

X_b = merged.copy()
for col in augmented_features:
    X_b[col] = X_b[col].fillna(augmented_medians[col])
for col in continuous_new:
    X_b[col] = X_b[col].fillna(new_feature_medians[col])
X_b_q = X_b.copy()
X_b_q[scaling_cols_b] = X_b_q[scaling_cols_b].astype("float64")
X_b_q[scaling_cols_b] = qt.transform(X_b[scaling_cols_b])
X_b_q_final = X_b_q[final_feature_cols_b].astype("float32")
x_infer = torch.tensor(X_b_q_final.values, dtype=torch.float32)
assert x_infer.shape == (203769, 186)
print(f"[PASS] Feature matrix rebuilt using SAVED preprocessing only. Shape: {x_infer.shape}")

col_index_map = {c: i for i, c in enumerate(final_feature_cols_b)}


SECTION 2: REBUILD GRAPH + FEATURE MATRIX


Graph: 203769 nodes, 234355 edges. Train mask: 29894. Class-3: 157205.


[PASS] Feature matrix rebuilt using SAVED preprocessing only. Shape: torch.Size([203769, 186])


## 3. Score-saturation investigation (Question 1)

Runs the frozen model once on the full graph to expose raw logits (both classes) alongside the
already-produced softmax scores. The existing `illicit_risk_score` values are **not** altered —
this only exposes the logits that produced them, and includes a sanity check that the recomputed
score matches the frozen CSV score exactly.

In [4]:
print("\n" + "="*70); print("SECTION 3: SCORE SATURATION -- LOGITS FOR TOP 20 + FULL CLASS-3"); print("="*70)

with torch.no_grad():
    full_logits = model(x_infer, edge_index)
assert torch.isfinite(full_logits).all().item(), "Non-finite logits!"
full_probs = F.softmax(full_logits, dim=1)[:, 1].numpy()
full_licit_logit = full_logits[:, 0].numpy()
full_illicit_logit = full_logits[:, 1].numpy()

risk_scores_csv = pd.read_csv(os.path.join(OUTPUTS_DIR, "unknown_transaction_risk_scores.csv"))
top20_scores_csv = pd.read_csv(os.path.join(OUTPUTS_DIR, "top_unknown_forensic_cases.csv"))

txid_to_idx = {tid: i for i, tid in enumerate(tx_data["txId"].values)}
top20_idx = [txid_to_idx[t] for t in top20_scores_csv["txId"].values]

logit_table = pd.DataFrame({
    "rank": top20_scores_csv["rank"].values,
    "txId": top20_scores_csv["txId"].values,
    "Time step": top20_scores_csv["Time step"].values,
    "illicit_logit": full_illicit_logit[top20_idx],
    "licit_logit": full_licit_logit[top20_idx],
    "illicit_risk_score": full_probs[top20_idx],
})
logit_table["logit_margin"] = logit_table["illicit_logit"] - logit_table["licit_logit"]

# sanity: recomputed score must match the frozen CSV score exactly (we are only exposing logits)
existing_scores = top20_scores_csv.set_index("txId")["illicit_risk_score"]
recomputed_scores = logit_table.set_index("txId")["illicit_risk_score"]
max_score_diff = (existing_scores - recomputed_scores).abs().max()
print(f"Max |existing score - recomputed score| for top 20: {max_score_diff:.2e} (must be ~0 -- scores unchanged, only logits newly exposed)")
assert max_score_diff < 1e-5

print("\nTop-20 logits/margins table:")
print(logit_table.round(6).to_string(index=False))


SECTION 3: SCORE SATURATION -- LOGITS FOR TOP 20 + FULL CLASS-3


Max |existing score - recomputed score| for top 20: 1.92e-08 (must be ~0 -- scores unchanged, only logits newly exposed)

Top-20 logits/margins table:
 rank      txId  Time step  illicit_logit  licit_logit  illicit_risk_score  logit_margin
    1  99942047         42       5.662352   -13.605924                 1.0     19.268276
    2  99942248         42       5.583810   -13.424760                 1.0     19.008570
    3  99942236         42       5.617908   -15.497320                 1.0     21.115229
    4  99942037         42       6.052202   -14.270629                 1.0     20.322830
    5  99943929         42       3.239440   -13.757377                 1.0     16.996817
    6  99942297         42       5.948198   -14.724738                 1.0     20.672935
    7  99942066         42       5.450463   -11.159943                 1.0     16.610405
    8 403235564         28       5.668222   -10.216146                 1.0     15.884367
    9 298977429         30       5.548558   -10.

## 4. Determine whether 1.0 is true saturation

Full decimal precision is printed for every top-20 score, and the logits are checked for
identity vs. genuine numerical difference. The reason is reported empirically — not assumed.

In [5]:
print("\nFull decimal precision (repr) for top 20 scores:")
for _, r in logit_table.iterrows():
    print(f"  txId={r['txId']}: repr(score)={repr(float(r['illicit_risk_score']))}  score:.12f={r['illicit_risk_score']:.12f}  illicit_logit={r['illicit_logit']:.6f}  licit_logit={r['licit_logit']:.6f}")

n_ge_0999_top20 = int((logit_table["illicit_risk_score"] >= 0.999).sum())
n_eq1_displayed_top20 = int((logit_table["illicit_risk_score"].round(4) == 1.0).sum())
min_logit_top20 = float(logit_table["illicit_logit"].min())
max_logit_top20 = float(logit_table["illicit_logit"].max())
min_margin_top20 = float(logit_table["logit_margin"].min())
max_margin_top20 = float(logit_table["logit_margin"].max())
n_identical_logits = int(logit_table["illicit_logit"].round(6).duplicated().sum())

print(f"\nTop-20 cases with score >= 0.999: {n_ge_0999_top20} / 20")
print(f"Top-20 cases with score == 1.0 at 4-decimal display precision: {n_eq1_displayed_top20} / 20")
print(f"Min/max illicit logit (top 20): {min_logit_top20:.6f} / {max_logit_top20:.6f}")
print(f"Min/max logit margin (top 20): {min_margin_top20:.6f} / {max_margin_top20:.6f}")
print(f"Duplicate (6-decimal-rounded) illicit logits among top 20: {n_identical_logits}")
print("\nFINDING: " + ("Logits are IDENTICAL across top-20 -> true saturation at the logit level."
      if n_identical_logits == 19 else
      "Logits DIFFER meaningfully across top-20 despite displayed scores rounding to 1.0 -- "
      "this is softmax/display saturation, not identical model outputs."))

# Full Class-3 saturation stats
class3_scores_all = full_probs[class3_mask_np]
total_class3 = int(class3_mask_np.sum())
count_eq1_displayed = int(pd.Series(class3_scores_all).round(4).eq(1.0).sum())
count_ge_0999 = int((class3_scores_all >= 0.999).sum())
count_ge_099 = int((class3_scores_all >= 0.99).sum())
print(f"\nFull Class-3 population (n={total_class3}):")
print(f"  score == 1.0 at 4-decimal display precision: {count_eq1_displayed}")
print(f"  score >= 0.999: {count_ge_0999}")
print(f"  score >= 0.99:  {count_ge_099}")


Full decimal precision (repr) for top 20 scores:
  txId=99942047.0: repr(score)=1.0  score:.12f=1.000000000000  illicit_logit=5.662352  licit_logit=-13.605924
  txId=99942248.0: repr(score)=1.0  score:.12f=1.000000000000  illicit_logit=5.583810  licit_logit=-13.424760
  txId=99942236.0: repr(score)=1.0  score:.12f=1.000000000000  illicit_logit=5.617908  licit_logit=-15.497320
  txId=99942037.0: repr(score)=1.0  score:.12f=1.000000000000  illicit_logit=6.052202  licit_logit=-14.270629
  txId=99943929.0: repr(score)=1.0  score:.12f=1.000000000000  illicit_logit=3.239440  licit_logit=-13.757377
  txId=99942297.0: repr(score)=1.0  score:.12f=1.000000000000  illicit_logit=5.948198  licit_logit=-14.724738
  txId=99942066.0: repr(score)=0.9999998807907104  score:.12f=0.999999880791  illicit_logit=5.450463  licit_logit=-11.159943
  txId=403235564.0: repr(score)=0.9999998807907104  score:.12f=0.999999880791  illicit_logit=5.668222  licit_logit=-10.216146
  txId=298977429.0: repr(score)=0.99999

## 5. Select representative cases

A deterministic, documented selection algorithm (not chosen to reach a desired conclusion):
- **A.** Top 5 by `logit_margin`.
- **B.** Up to 3 cases with the largest standardized (z-score) distance from the top-20 centroid
  across `[FIN5, total_BTC, fees, NET2, TMP5]` ("unusual behavioural profile"), skipping cases
  already selected.
- **C.** Guarantees at least one Time step 42 case (already satisfied by A here).
- **D.** Guarantees at least one Time step 46 case.
- **E.** Adds the highest-`FIN5` (fee-to-value ratio) case if not already present ("unusual
  financial behaviour").

In [6]:
print("\n" + "="*70); print("SECTION 5: SELECT REPRESENTATIVE CASES"); print("="*70)

merged_indexed = merged_raw.set_index("txId")
top20_full = logit_table.merge(merged_indexed.drop(columns=[c for c in ["Time step"] if c in merged_indexed.columns]).reset_index(), on="txId", how="left")

selected = {}
reasons = {}

# A. top 5 by logit margin
top5_margin = top20_full.sort_values("logit_margin", ascending=False).head(5)
for _, r in top5_margin.iterrows():
    selected[r["txId"]] = r
    reasons.setdefault(r["txId"], []).append("top-5 logit margin")

# B. 2-3 unusual behavioural profiles: z-score distance from top-20 centroid on a small feature set
profile_cols_for_outlier = ["FIN5", "total_BTC", "fees", "NET2", "TMP5"]
prof = top20_full[profile_cols_for_outlier].astype(float)
z = (prof - prof.mean()) / prof.std(ddof=0).replace(0, 1)
dist = np.sqrt((z**2).sum(axis=1))
top20_full["_outlier_dist"] = dist.values
outlier_candidates = top20_full.sort_values("_outlier_dist", ascending=False)
n_added_B = 0
for _, r in outlier_candidates.iterrows():
    if n_added_B >= 3:
        break
    if r["txId"] not in selected:
        selected[r["txId"]] = r
        reasons.setdefault(r["txId"], []).append(f"unusual behavioural profile (z-distance={r['_outlier_dist']:.2f})")
        n_added_B += 1

# C. at least one case at Time step 42
if not any(top20_full.loc[top20_full["txId"] == t, "Time step"].iloc[0] == 42 for t in selected):
    cand = top20_full[top20_full["Time step"] == 42].sort_values("logit_margin", ascending=False)
    if len(cand):
        r = cand.iloc[0]
        selected[r["txId"]] = r
        reasons.setdefault(r["txId"], []).append("coverage: Time step 42")

# D. at least one case at Time step 46
if not any(top20_full.loc[top20_full["txId"] == t, "Time step"].iloc[0] == 46 for t in selected):
    cand = top20_full[top20_full["Time step"] == 46].sort_values("logit_margin", ascending=False)
    if len(cand):
        r = cand.iloc[0]
        selected[r["txId"]] = r
        reasons.setdefault(r["txId"], []).append("coverage: Time step 46")

# E. unusual financial behaviour (largest FIN5 fee-to-value ratio among top 20), if not already covered
fin5_outlier = top20_full.sort_values("FIN5", ascending=False).iloc[0]
if fin5_outlier["txId"] not in selected:
    selected[fin5_outlier["txId"]] = fin5_outlier
    reasons.setdefault(fin5_outlier["txId"], []).append(f"unusual financial behaviour (highest FIN5={fin5_outlier['FIN5']:.4g} among top 20)")
else:
    reasons[fin5_outlier["txId"]].append(f"also highest FIN5={fin5_outlier['FIN5']:.4g} among top 20")

representative_txids = list(selected.keys())
print(f"Selected {len(representative_txids)} representative cases:")
for t in representative_txids:
    print(f"  txId={t}  Time step={selected[t]['Time step']}  logit_margin={selected[t]['logit_margin']:.4f}  reasons={reasons[t]}")


SECTION 5: SELECT REPRESENTATIVE CASES


Selected 8 representative cases:
  txId=99942236  Time step=42  logit_margin=21.1152  reasons=['top-5 logit margin']
  txId=99942297  Time step=42  logit_margin=20.6729  reasons=['top-5 logit margin']
  txId=99942037  Time step=42  logit_margin=20.3228  reasons=['top-5 logit margin']
  txId=99942047  Time step=42  logit_margin=19.2683  reasons=['top-5 logit margin']
  txId=99942248  Time step=42  logit_margin=19.0086  reasons=['top-5 logit margin']
  txId=196098540  Time step=46  logit_margin=15.0539  reasons=['unusual behavioural profile (z-distance=6.22)']
  txId=15340246  Time step=46  logit_margin=14.8170  reasons=['unusual behavioural profile (z-distance=4.46)', 'also highest FIN5=1.017 among top 20']
  txId=191501490  Time step=8  logit_margin=15.2621  reasons=['unusual behavioural profile (z-distance=2.98)']


## 6. Train-only occlusion baseline

The perturbation reference is derived **only from training rows** in the already-transformed
(QuantileTransformer) feature space: the **median** for each of the 183 scaled columns, and the
**mode** for each of the 3 binary columns. Nothing is fit or refit — this is a statistic computed
once over already-saved, already-transformed values. Class-3 data is never used to define this
baseline.

In [7]:
print("\n" + "="*70); print("SECTION 6: TRAIN-ONLY OCCLUSION BASELINE (no new fitting)"); print("="*70)

baseline_vector = np.zeros(186, dtype=np.float32)
x_infer_np_train = x_infer.numpy()[train_mask_np]
for c in scaling_cols_b:
    idx = col_index_map[c]
    baseline_vector[idx] = np.median(x_infer_np_train[:, idx])
for c in binary_cols_b:
    idx = col_index_map[c]
    vals, counts = np.unique(x_infer_np_train[:, idx], return_counts=True)
    baseline_vector[idx] = vals[np.argmax(counts)]
print("Baseline reference vector computed from TRAIN rows only (median for scaled cols, mode for binary cols).")
print(f"Example: baseline['Local_feature_1']={baseline_vector[col_index_map['Local_feature_1']]:.4f}  "
      f"baseline['FIN3']={baseline_vector[col_index_map['FIN3']]:.4f}  "
      f"baseline['STR2']={baseline_vector[col_index_map['STR2']]:.4f}")


SECTION 6: TRAIN-ONLY OCCLUSION BASELINE (no new fitting)
Baseline reference vector computed from TRAIN rows only (median for scaled cols, mode for binary cols).
Example: baseline['Local_feature_1']=-0.0037  baseline['FIN3']=0.0000  baseline['STR2']=0.0000


## 7. Feature-occlusion attribution + graph-neighbourhood separation

**Attribution method:** single-feature occlusion. For each representative transaction and each of
the 186 features, the target node's value for that feature is temporarily replaced by the
train-only baseline, a full-graph forward pass is run, the change in the target node's illicit
logit is recorded, and the original value is restored immediately.

**Sign convention (read carefully):** `attribution_delta = logit_after_replacing_with_baseline -
logit_with_original_value`. A **negative** delta means the transaction's own (original) value
pushes the illicit-class logit **higher** than the typical/baseline value would — i.e. the actual
value is contributing toward the high score. A **positive** delta means the actual value pushes the
logit **lower** (more licit-like) than the baseline.

**Target-node vs. graph-neighbourhood effect:** because this is a 2-layer GraphSAGE model using the
*same* `edge_index` in both conv layers, removing all edges directed **into** the target node
(`dst == target`) cuts off neighbour aggregation at both layers for that node, isolating its output
to a self-only transform of its own features. Comparing this "isolated" prediction to the full-graph
prediction gives an approximate **graph-neighbourhood effect**. This is a proxy, not an exact
decomposition — see the limitations section.

In [8]:
print("\n" + "="*70); print("SECTION 7: FEATURE-OCCLUSION ATTRIBUTION (full graph)"); print("="*70)

FINANCIAL_SET = set(["FIN3","FIN4","FIN5","FIN6","total_BTC","fees","size",
                      "in_BTC_min","in_BTC_max","in_BTC_mean","in_BTC_median","in_BTC_total",
                      "out_BTC_min","out_BTC_max","out_BTC_mean","out_BTC_median","out_BTC_total"])
STRUCTURAL_SET = set(["STR1","STR2","num_input_addresses","num_output_addresses","in_txs_degree","out_txs_degree"])
TEMPORAL_SET = set(["TMP1","TMP2","TMP4","TMP5","TMP6","TMP7","TMP6_is_first_appearance"])
NETWORK_SET = set(["NET1","NET2"])

def categorize(feat):
    if feat in FINANCIAL_SET: return "FINANCIAL"
    if feat in STRUCTURAL_SET: return "STRUCTURAL"
    if feat in TEMPORAL_SET: return "TEMPORAL"
    if feat in NETWORK_SET: return "NETWORK"
    return "ORIGINAL_INHERITED"

attribution_rows = []
neighbourhood_rows = []
n_cases = len(representative_txids)
case_i = 0
for txid in representative_txids:
    case_i += 1
    target_idx = txid_to_idx[txid]
    row_info = selected[txid]
    log(f"Case {case_i}/{n_cases}: txId={txid} (node {target_idx}) -- running 186-feature occlusion...")

    baseline_illicit_logit = float(full_illicit_logit[target_idx])
    baseline_licit_logit = float(full_licit_logit[target_idx])
    baseline_score = float(full_probs[target_idx])

    original_row = x_infer[target_idx].clone()
    for f_idx, feat_name in enumerate(final_feature_cols_b):
        orig_val = float(original_row[f_idx])
        pert_val = float(baseline_vector[f_idx])
        if abs(orig_val - pert_val) < 1e-9:
            attribution_rows.append({
                "txId": txid, "Time step": row_info["Time step"], "illicit_risk_score": baseline_score,
                "illicit_logit": baseline_illicit_logit, "licit_logit": baseline_licit_logit,
                "logit_margin": baseline_illicit_logit - baseline_licit_logit,
                "feature": feat_name, "feature_category": categorize(feat_name),
                "original_value": orig_val, "perturbed_value": pert_val,
                "attribution_delta": 0.0, "absolute_attribution": 0.0,
            })
            continue
        x_infer[target_idx, f_idx] = pert_val
        with torch.no_grad():
            pert_logits_row = model(x_infer, edge_index)[target_idx]
        x_infer[target_idx, f_idx] = orig_val  # restore immediately
        pert_illicit_logit = float(pert_logits_row[1])
        delta = pert_illicit_logit - baseline_illicit_logit
        attribution_rows.append({
            "txId": txid, "Time step": row_info["Time step"], "illicit_risk_score": baseline_score,
            "illicit_logit": baseline_illicit_logit, "licit_logit": baseline_licit_logit,
            "logit_margin": baseline_illicit_logit - baseline_licit_logit,
            "feature": feat_name, "feature_category": categorize(feat_name),
            "original_value": orig_val, "perturbed_value": pert_val,
            "attribution_delta": delta, "absolute_attribution": abs(delta),
        })
    assert torch.allclose(x_infer[target_idx], original_row), "Feature row not fully restored!"

    # Neighbourhood effect: remove all edges INTO the target node, rerun
    keep_mask = edge_index[1] != target_idx
    edge_index_isolated = edge_index[:, keep_mask]
    with torch.no_grad():
        isolated_logits_row = model(x_infer, edge_index_isolated)[target_idx]
    isolated_illicit_logit = float(isolated_logits_row[1])
    isolated_licit_logit = float(isolated_logits_row[0])
    neighbourhood_rows.append({
        "txId": txid, "Time step": row_info["Time step"],
        "full_graph_illicit_logit": baseline_illicit_logit,
        "isolated_illicit_logit": isolated_illicit_logit,
        "neighbourhood_effect_delta": baseline_illicit_logit - isolated_illicit_logit,
        "full_graph_score": baseline_score,
        "isolated_score": float(torch.softmax(isolated_logits_row, dim=0)[1]),
        "n_incoming_edges_removed": int((~keep_mask).sum()),
    })

log("Feature occlusion + neighbourhood isolation complete for all representative cases.")
attribution_df = pd.DataFrame(attribution_rows)
neighbourhood_df = pd.DataFrame(neighbourhood_rows)
assert np.isfinite(attribution_df["attribution_delta"].values).all(), "Non-finite attribution values!"
assert np.isfinite(attribution_df["illicit_logit"].values).all()
print(f"[PASS] All attribution deltas and logits finite. Total attribution rows: {len(attribution_df)}")

attribution_df = attribution_df.sort_values(["txId", "absolute_attribution"], ascending=[True, False]).reset_index(drop=True)
attribution_path = os.path.join(OUTPUTS_DIR, "top_forensic_model_attributions.csv")
attribution_df.to_csv(attribution_path, index=False)
print(f"Saved: {attribution_path}  ({len(attribution_df)} rows)")

print("\nNeighbourhood effect (full graph vs. isolated, target node's incoming edges removed):")
print(neighbourhood_df.round(6).to_string(index=False))


SECTION 7: FEATURE-OCCLUSION ATTRIBUTION (full graph)
[   19.9s] Case 1/8: txId=99942236 (node 75864) -- running 186-feature occlusion...


[   54.7s] Case 2/8: txId=99942297 (node 75868) -- running 186-feature occlusion...


[   89.0s] Case 3/8: txId=99942037 (node 75852) -- running 186-feature occlusion...


[  123.2s] Case 4/8: txId=99942047 (node 75855) -- running 186-feature occlusion...


[  155.8s] Case 5/8: txId=99942248 (node 75865) -- running 186-feature occlusion...


[  188.9s] Case 6/8: txId=196098540 (node 118113) -- running 186-feature occlusion...


[  221.7s] Case 7/8: txId=15340246 (node 13869) -- running 186-feature occlusion...


[  255.7s] Case 8/8: txId=191501490 (node 115204) -- running 186-feature occlusion...


[  288.2s] Feature occlusion + neighbourhood isolation complete for all representative cases.


[PASS] All attribution deltas and logits finite. Total attribution rows: 1488
Saved: C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\top_forensic_model_attributions.csv  (1488 rows)

Neighbourhood effect (full graph vs. isolated, target node's incoming edges removed):
     txId  Time step  full_graph_illicit_logit  isolated_illicit_logit  neighbourhood_effect_delta  full_graph_score  isolated_score  n_incoming_edges_removed
 99942236         42                  5.617908                5.862126                   -0.244218               1.0        1.000000                         1
 99942297         42                  5.948198                5.550660                    0.397538               1.0        1.000000                         1
 99942037         42                  6.052202                5.345914                    0.706287               1.0        1.000000                         1
 99942047         42                  5.662352                5.117660           

## 8. Fingerprint feature attribution check

Empirically checks whether `num_input_addresses`, `num_output_addresses`,
`TMP6_is_first_appearance`, `STR1`, `STR2` — the features behind the previously-observed
1-in/1-out/first-appearance pattern — actually show measurable model sensitivity, rather than
assuming they do.

In [9]:
print("\n" + "="*70); print("SECTION 8: FINGERPRINT FEATURE ATTRIBUTION CHECK"); print("="*70)

fingerprint_feats = ["num_input_addresses", "num_output_addresses", "TMP6_is_first_appearance", "STR1", "STR2"]
fp_check = attribution_df[attribution_df["feature"].isin(fingerprint_feats)][
    ["txId", "feature", "original_value", "perturbed_value", "attribution_delta", "absolute_attribution"]
]
print(fp_check.to_string(index=False))
mean_abs_fp = fp_check["absolute_attribution"].mean()
mean_abs_all = attribution_df["absolute_attribution"].mean()
print(f"\nMean |attribution| for fingerprint features: {mean_abs_fp:.4f}")
print(f"Mean |attribution| across ALL 186 features (all representative cases): {mean_abs_all:.4f}")
print("\nNote: STR1, STR2, and TMP6_is_first_appearance show attribution_delta == 0 for every")
print("representative case because their ORIGINAL value already equals the train-only baseline")
print("(median/mode) -- i.e. these transactions are NOT unusual on those specific features relative")
print("to the training population, so a median/mode-baseline occlusion cannot reveal sensitivity for")
print("them. num_input_addresses (and, more weakly, num_output_addresses) do show real, consistently")
print("negative attribution across all 8 cases, meaning their actual values measurably push the")
print("illicit logit higher than the training-typical value would.")


SECTION 8: FINGERPRINT FEATURE ATTRIBUTION CHECK
     txId                  feature  original_value  perturbed_value  attribution_delta  absolute_attribution
 15340246      num_input_addresses       -5.199337         0.295296          -0.203605              0.203605
 15340246     num_output_addresses       -5.199337        -0.011291          -0.055005              0.055005
 15340246                     STR1        0.304482         0.304482           0.000000              0.000000
 15340246                     STR2        0.000000         0.000000           0.000000              0.000000
 15340246 TMP6_is_first_appearance        1.000000         1.000000           0.000000              0.000000
 99942037      num_input_addresses       -5.199337         0.295296          -0.202909              0.202909
 99942037     num_output_addresses       -5.199337        -0.011291          -0.082958              0.082958
 99942037                     STR1        0.304482         0.304482           

## 9. Time-step / temporal attribution

Reports `Time step` and all seven temporal features' attribution for the representative cases. If
`Time step` shows strong attribution, that is noted as a possible temporal-distribution effect, not
evidence of fraud.

In [10]:
print("\n" + "="*70); print("SECTION 9: TIME-STEP / TEMPORAL ATTRIBUTION"); print("="*70)

ts_feat_check = attribution_df[attribution_df["feature"].isin(
    ["Time step","TMP1","TMP2","TMP4","TMP5","TMP6","TMP7","TMP6_is_first_appearance"])][
    ["txId","Time step","feature","original_value","attribution_delta","absolute_attribution"]
]
print(ts_feat_check.to_string(index=False))
print("\nTime step's own attribution is consistently small and negative across cases (roughly -0.03")
print("to -0.14 logits), notably smaller in magnitude than TMP5 or the top financial/structural")
print("features. This is reported as a modest, possibly temporal-distribution-related effect --")
print("not treated as fraud evidence.")


SECTION 9: TIME-STEP / TEMPORAL ATTRIBUTION
     txId  Time step                  feature  original_value  attribution_delta  absolute_attribution
 15340246         46                     TMP5        1.027247          -0.355253              0.355253
 15340246         46                     TMP2        1.844811           0.261495              0.261495
 15340246         46                Time step        5.199337          -0.103542              0.103542
 15340246         46                     TMP1       -5.199337           0.000000              0.000000
 15340246         46                     TMP4       -5.199337           0.000000              0.000000
 15340246         46                     TMP6       -5.199337           0.000000              0.000000
 15340246         46                     TMP7       -5.199337           0.000000              0.000000
 15340246         46 TMP6_is_first_appearance        1.000000           0.000000              0.000000
 99942037         42        

## 10. Case-level attribution summary

Top-3 attributed features per behavioural category for each representative case, saved to
`outputs/forensic_case_attribution_summary.csv`.

In [11]:
print("\n" + "="*70); print("SECTION 10: CASE-LEVEL ATTRIBUTION SUMMARY"); print("="*70)

def top_features_str(df_case, category, k=3):
    sub = df_case[df_case["feature_category"] == category].sort_values("absolute_attribution", ascending=False).head(k)
    return "; ".join(f"{r['feature']}({r['attribution_delta']:+.3f})" for _, r in sub.iterrows())

summary_rows = []
for txid in representative_txids:
    dfc = attribution_df[attribution_df["txId"] == txid]
    row_info = selected[txid]
    summary_rows.append({
        "txId": txid,
        "Time step": row_info["Time step"],
        "illicit_risk_score": float(row_info["illicit_risk_score"]),
        "logit_margin": float(row_info["logit_margin"]),
        "top_financial_features": top_features_str(dfc, "FINANCIAL"),
        "top_structural_features": top_features_str(dfc, "STRUCTURAL"),
        "top_temporal_features": top_features_str(dfc, "TEMPORAL"),
        "top_network_features": top_features_str(dfc, "NETWORK"),
    })
summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(OUTPUTS_DIR, "forensic_case_attribution_summary.csv")
summary_df.to_csv(summary_path, index=False)
print(summary_df.to_string(index=False))
print(f"\nSaved: {summary_path}  ({len(summary_df)} rows)")


SECTION 10: CASE-LEVEL ATTRIBUTION SUMMARY
     txId  Time step  illicit_risk_score  logit_margin                            top_financial_features                                                           top_structural_features                    top_temporal_features       top_network_features
 99942236         42                 1.0     21.115229          FIN6(-0.279); FIN3(-0.135); size(-0.113) num_input_addresses(-0.192); out_txs_degree(-0.073); num_output_addresses(-0.065) TMP5(-0.381); TMP1(+0.000); TMP2(+0.000) NET2(-0.020); NET1(+0.000)
 99942297         42                 1.0     20.672935          FIN6(-0.251); size(-0.155); FIN3(-0.134) num_input_addresses(-0.178); num_output_addresses(-0.052); out_txs_degree(-0.019) TMP5(-0.355); TMP1(+0.000); TMP2(+0.000) NET2(-0.018); NET1(+0.000)
 99942037         42                 1.0     20.322830          FIN6(-0.302); FIN3(-0.132); size(-0.114) num_input_addresses(-0.203); num_output_addresses(-0.083); out_txs_degree(-0.068) TMP5

## 11. Score-saturation summary table

In [12]:
print("\n" + "="*70); print("SECTION 11: SAVE class3_score_saturation_summary.csv"); print("="*70)

saturation_summary = pd.DataFrame([{
    "total_class3": total_class3,
    "count_score_exactly_1_displayed": count_eq1_displayed,
    "count_score_ge_0_999": count_ge_0999,
    "count_score_ge_0_99": count_ge_099,
    "min_illicit_logit_top20": min_logit_top20,
    "max_illicit_logit_top20": max_logit_top20,
    "min_logit_margin_top20": min_margin_top20,
    "max_logit_margin_top20": max_margin_top20,
}])
saturation_path = os.path.join(OUTPUTS_DIR, "class3_score_saturation_summary.csv")
saturation_summary.to_csv(saturation_path, index=False)
print(saturation_summary.to_string(index=False))
print(f"Saved: {saturation_path}")


SECTION 11: SAVE class3_score_saturation_summary.csv
 total_class3  count_score_exactly_1_displayed  count_score_ge_0_999  count_score_ge_0_99  min_illicit_logit_top20  max_illicit_logit_top20  min_logit_margin_top20  max_logit_margin_top20
       157205                              579                  2222                 5320                 1.772007                 6.065658               14.817047               21.115229
Saved: C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\class3_score_saturation_summary.csv


## 12. Visualizations

Three plots only: the Class-3 score distribution, top attribution magnitudes per representative
case, and logit margin vs. displayed score for the top 20 (since the probability scale hides real
differences once scores round to 1.0).

In [13]:
print("\n" + "="*70); print("SECTION 12: VISUALIZATIONS"); print("="*70)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(class3_scores_all, bins=60, color="#4C72B0", edgecolor="none")
ax.set_yscale("log")
ax.set_xlabel("illicit_risk_score")
ax.set_ylabel("count (log scale)")
ax.set_title(f"Distribution of Class-3 illicit-risk scores (n={total_class3})")
fig.tight_layout()
p1 = os.path.join(PLOTS_DIR, "class3_score_distribution.png")
fig.savefig(p1, dpi=130)
plt.close(fig)
print(f"Saved: {p1}")

n_cols = 3
n_rows = int(np.ceil(n_cases / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 3.2*n_rows))
axes = np.atleast_1d(axes).flatten()
for i, txid in enumerate(representative_txids):
    dfc = attribution_df[attribution_df["txId"] == txid].sort_values("absolute_attribution", ascending=False).head(10)
    ax = axes[i]
    colors = ["#C44E52" if d < 0 else "#55A868" for d in dfc["attribution_delta"]]
    ax.barh(dfc["feature"][::-1], dfc["attribution_delta"][::-1], color=colors[::-1])
    ax.set_title(f"txId {txid} (t={selected[txid]['Time step']})", fontsize=9)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.tick_params(labelsize=7)
for j in range(len(representative_txids), len(axes)):
    axes[j].axis("off")
fig.suptitle("Top-10 |attribution| features per representative case (logit delta)", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.96])
p2 = os.path.join(PLOTS_DIR, "top_feature_attributions.png")
fig.savefig(p2, dpi=130)
plt.close(fig)
print(f"Saved: {p2}")

fig, ax1 = plt.subplots(figsize=(9, 4.5))
order = logit_table.sort_values("rank")
x = np.arange(len(order))
ax1.bar(x, order["logit_margin"], color="#4C72B0", alpha=0.8, label="logit_margin")
ax1.set_xlabel("top-20 rank")
ax1.set_ylabel("logit_margin (illicit - licit)", color="#4C72B0")
ax1.set_xticks(x)
ax1.set_xticklabels(order["rank"])
ax2 = ax1.twinx()
ax2.plot(x, order["illicit_risk_score"], color="#C44E52", marker="o", label="illicit_risk_score")
ax2.set_ylabel("illicit_risk_score", color="#C44E52")
ax2.set_ylim(0.98, 1.002)
fig.suptitle("Logit margin vs. displayed risk score (top 20) -- probability scale hides differences")
fig.tight_layout()
p3 = os.path.join(PLOTS_DIR, "logit_margin_vs_score.png")
fig.savefig(p3, dpi=130)
plt.close(fig)
print(f"Saved: {p3}")


SECTION 12: VISUALIZATIONS


Saved: C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\forensic_analysis_plots\class3_score_distribution.png


Saved: C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\forensic_analysis_plots\top_feature_attributions.png


Saved: C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\forensic_analysis_plots\logit_margin_vs_score.png


## 13. Limitations

1. **Class 3 labels are unknown.** No Class-3 transaction has a verified ground-truth label
   anywhere in this project.
2. **The model score is not a calibrated ground-truth probability.** It is a softmax output of a
   discriminatively-trained classifier, not a validated real-world likelihood.
3. **Score saturation can make probability values less informative at the upper end.** As shown
   above, illicit logits ranging from ~1.77 to ~6.07 (and margins from ~14.8 to ~21.1) all map to
   displayed scores of 1.0000 — the probability scale hides real, substantial differences between
   these transactions.
4. **GraphSAGE predictions depend on neighbourhood message passing.** A node's output reflects both
   its own features and information aggregated from its graph neighbourhood; the isolation
   experiment above is an approximate way to separate these, not an exact decomposition.
5. **Feature perturbation measures model sensitivity, not causal importance.** A large
   attribution delta shows the model's output is sensitive to that feature's value relative to the
   chosen baseline — it does not establish that the feature *causes* illicit behaviour.
6. **Correlated features can make individual attribution unstable.** Several of the 186 features
   (e.g. the many inherited `Local_feature_*` columns, or `num_input_addresses`/`STR1`) are likely
   correlated; occluding one at a time does not account for these dependencies, and a feature that
   shows zero attribution here (e.g. `STR1`, `TMP6_is_first_appearance`) may simply already equal
   the training-typical baseline for these specific transactions, not be unimportant in general.
7. **The 1-input/1-output/first-appearance fingerprint is an observed pattern and should not be
   interpreted as proof of illicit activity.** It is common in the labelled reference population
   too (61.71% first-appearance, 52.83% single-input) even though the *combination* is rarer (7.93%).
8. **Reference-population comparisons (from the prior notebook) are descriptive.** They show
   association with labelled classes, not causal or forensic proof for Class-3 transactions.
9. **This analysis does not establish real-world criminal or illicit status for any transaction.**
   All findings are model-internal diagnostics presented to support, not replace, human forensic
   judgement.

## 14. Final verification and completion summary

In [14]:
print("\n" + "="*70); print("FINAL VERIFICATION"); print("="*70)

for txid in representative_txids:
    assert class3_mask_np[txid_to_idx[txid]], f"txId {txid} is not a valid Class-3 transaction!"
print("[PASS] All selected representative txIds are valid Class-3 transactions.")
assert np.isfinite(attribution_df["attribution_delta"].values).all()
assert np.isfinite(logit_table["illicit_logit"].values).all()
print("[PASS] All attribution deltas and logits are finite.")

print("\nMODEL ATTRIBUTION AND SCORE-SATURATION ANALYSIS COMPLETE")
print(f"1. Saturation: {'TRUE saturation (identical logits)' if n_identical_logits==19 else 'DISPLAY/SOFTMAX saturation -- logits differ meaningfully'}")
print(f"2. Top-20 logits differ despite displayed score 1.0: {19-n_identical_logits}/19 pairs distinguishable")
print(f"3. Representative transactions analyzed: {len(representative_txids)} -- {representative_txids}")
print(f"4. Files created:")
print(f"   - {attribution_path}")
print(f"   - {summary_path}")
print(f"   - {saturation_path}")
print(f"   - {p1}")
print(f"   - {p2}")
print(f"   - {p3}")
print(f"\nTotal runtime: {time.time()-t0:.1f}s")


FINAL VERIFICATION
[PASS] All selected representative txIds are valid Class-3 transactions.
[PASS] All attribution deltas and logits are finite.

MODEL ATTRIBUTION AND SCORE-SATURATION ANALYSIS COMPLETE
1. Saturation: DISPLAY/SOFTMAX saturation -- logits differ meaningfully
2. Top-20 logits differ despite displayed score 1.0: 19/19 pairs distinguishable
3. Representative transactions analyzed: 8 -- [99942236, 99942297, 99942037, 99942047, 99942248, 196098540, 15340246, 191501490]
4. Files created:
   - C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\top_forensic_model_attributions.csv
   - C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\forensic_case_attribution_summary.csv
   - C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\class3_score_saturation_summary.csv
   - C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\forensic_analysis_plots\class3_score_distribution.png
   - C:\Users\koush\OneDrive\Documents\Financial_Forensics\outpu